# Semana 7 — M1 Soluciones: APIs para Data Engineers
**Bootcamp:** Fundamentos de Ingeniería de Datos — Databricks SQL

### Ejercicio 1.1 — SETUP

In [0]:
import requests
import json

# GET request a DolarAPI — retorna una lista de JSONs planos
response = requests.get("https://dolarapi.com/v1/dolares")
print(f"Status code: {response.status_code}")

data = response.json()
print(f"Cotizaciones: {len(data)}")

for cotizacion in data:
    print(f"{cotizacion['nombre']:>15}: compra={cotizacion['compra']}, venta={cotizacion['venta']}")

### Ejercicio 1.2 — GUIDED

In [0]:
from pyspark.sql.functions import current_timestamp

# JSON plano → DataFrame directo con createDataFrame
df_dolar = spark.createDataFrame(data)

# Agregar timestamp de ingesta para saber cuándo se cargó
df_dolar_bronze = df_dolar.withColumn("ingesta_timestamp", current_timestamp())
df_dolar_bronze.write.format("delta").mode("overwrite").saveAsTable("bootcamp.bronze.cotizacion_dolar")

spark.table("bootcamp.bronze.cotizacion_dolar").show()

### Ejercicio 1.3 — INDEPENDENT

In [0]:
# OpenMeteo: JSON anidado — hay que desanidar hourly.time, hourly.temperature_2m, etc.
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -34.6037,
    "longitude": -58.3816,
    "hourly": "temperature_2m,wind_speed_10m",
    "forecast_days": 2
}
response = requests.get(url, params=params)
data_clima = response.json()

# Desanidar: recorrer las listas paralelas dentro de hourly
hourly = data_clima["hourly"]
registros = []
for i in range(len(hourly["time"])):
    registros.append({
        "timestamp": hourly["time"][i],
        "temperatura_c": hourly["temperature_2m"][i],
        "viento_kmh": hourly["wind_speed_10m"][i]
    })

df_clima = spark.createDataFrame(registros)
df_clima_bronze = df_clima.withColumn("ingesta_timestamp", current_timestamp())
df_clima_bronze.write.format("delta").mode("overwrite").saveAsTable("bootcamp.bronze.clima_api")

spark.table("bootcamp.bronze.clima_api").show(10)